In [ ]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
pd.set_option('display.max.column', None)
import os
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load and inspect the dataset

df = pd.read_excel(r"C:\Users\phabr\Downloads\Coffee Shop Sales.xlsx")
df.head(5)


In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
print(df.select_dtypes(include='number').describe()) 

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.drop_duplicates('transaction_id', inplace=True)

In [ ]:
df.head(10)

In [ ]:
# There are no missing and duplicated values in the dataset

In [ ]:
df['product_category'].unique()

In [ ]:
df['product_category'].nunique()

In [ ]:
df['store_location'].unique()

In [ ]:
df['product_type'].unique()

In [ ]:
df['product_type'].nunique()

In [ ]:
# Feature Engineering
data = df.copy()  # Duplicate the dataset

# Add some news columns
data['transaction_timestamp'] = data['transaction_date'].astype(str) + ' ' + data['transaction_time'].astype(str) 
data['transaction_timestamp'] = pd.to_datetime(data['transaction_timestamp'])      # Convert the data type to datetime
data['transaction_month'] = data['transaction_date'].dt.month          # Extract the month number from the transaction date column
data['transaction_monthname'] = data['transaction_date'].dt.month_name    # Extract the month name from the transaction date column
data['day_of_the_week'] = data['transaction_date'].dt.day_of_week     # Extract the day of the week from the transaction date column
data['day'] = data['transaction_date'].dt.day_name         # Extract the day name from the transaction date column
data['hour'] = data['transaction_timestamp'].dt.hour   # Extract the hour from the transaction date column
data['revenue'] = data['unit_price'] * data['transaction_qty']     # Add the revenue column = unit_price * transaction_qty

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
print(data['transaction_timestamp'].min())   # Check for earliest transaction date
print(data['transaction_timestamp'].max())   # Check for latest transaction date
print(data['transaction_timestamp'].value_counts().sort_index(ascending=True)) # Check the unique transaction hours from the smallest to largest
print(data['transaction_qty'].value_counts())  # Check the unique transactions quantities
print(data['store_id'].nunique()) # Check number of uniques stores
print(data['store_location'].unique()) # Check the unique store location
print(data['product_category'].nunique()) # Check number of uniques product_categories
print(data['product_type'].unique()) # Check the unique product_type

In [ ]:
# Data Analysis -- Calculating KPIs

Total_orders = data['transaction_id'].count()
print('Total_orders:' , Total_orders) # Number of transactions in the dataset

Quantity_sold = data['transaction_qty'].sum()
print('Number of units sold:',Quantity_sold) # Number of units sold 

No_of_days = (data['transaction_date'].max() - data['transaction_date'].min()).days
print('Number of days of transactions:', No_of_days) # Number of days of transactions

Avg_order_per_day = Total_orders / No_of_days 
print('Average order per day:', round(Avg_order_per_day,2)) # Average order per day

Total_revenue = data['revenue'].sum()
print('Total revenue generated from the sales:', round(Total_revenue,2)) # Total revenue generated from the sales

Avg_order_value = Total_revenue / Total_orders
print('Average revenue generated per order:',round(Avg_order_value,2)) # Average revenue generated per order

In [ ]:
# Orders Analysis 
# Find the total orders by transaction hours 
hourly_orders = data.groupby(['hour'], as_index=False).agg(Total_orders=('transaction_id', 'count'))
print(hourly_orders)

# Plot the chart for hourly orders
fig, ax = plt.subplots(figsize=[10,3])
ax.bar(x=hourly_orders['hour'].astype('str'), height=hourly_orders['Total_orders'])

ax.set_title('Hourly Orders') #Add title
ax.set_xlabel('Transaction Hours') #Add Xaxis Label
ax.spines[['top', 'right', 'left']].set_visible(False) #Remove spines
ax.yaxis.set_visible(False) #Remove Yaxis
plt.show()

In [ ]:
# Order by day of the week 

day_of_week = data.groupby(['day_of_the_week', 'day'], as_index=False).agg(Total_orders=('transaction_id', 'count'))
print(day_of_week)

# Plot a heatmap of the orders by day of the week and transaction hour

day_hour_orders = data.pivot_table(
    index= 'day_of_the_week',
    columns= 'hour',
    values= 'transaction_id',
    aggfunc= 'count'
)
print(day_hour_orders)

fig, ax = plt.subplots(figsize = [10,4])
sns.heatmap(day_hour_orders, cmap='Blues')
plt.show()